[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NguyenVu04/band-tilt/blob/main/notebooks/05a_bo.ipynb)

# 05a — Bayesian Optimization

**Purpose.** Search the absolute-tilt space with Ax/BoTorch and produce
`theta*_BO`. PROJECT.md section 20.

**The comparison discipline.** This notebook and 05b solve the *same* problem:
the same bounds from `src.optim.space`, the same KPIs from `src.kpi`, the same
surrogate. Anything else would compare two implementations rather than two
methods — see docs/adr/0006. Nothing about the problem is defined here.

**Two surrogates, and they are not the same thing.** The KPI surrogate from
notebook 04 replaces the *simulator*; the Gaussian process Ax fits online
replaces the *objective function* within the search. Running the search against
the KPI surrogate stacks two error sources, which is why section 7 re-validates
the best candidates with Sionna-RT.

**Requires** `uv sync --extra bo`.

## 0. Environment

Run this section first, wherever you are.

**Locally** it only walks up to the project root and makes it the working
directory, so the root-relative paths in `configs/data.yaml` resolve the same way
they do for `task clean:data` and the DVC pipeline. Nothing is installed.

**In Colab** it also clones the repository, puts it on `sys.path` so `import src`
works without an editable install, and installs the packages Colab does not ship.
Note that `data/` and `models/` are DVC-tracked and therefore *not* part of the
clone — a fresh runtime has neither. See the Drive cell below.

In [ ]:
# --- Environment bootstrap -------------------------------------------------
# Identical in every notebook. Forked the repository? Change these three values
# and the badge URL at the top of this notebook.
REPO_URL = "https://github.com/NguyenVu04/band-tilt.git"
BRANCH = "main"
SUBDIR = ""  # the project root is the repository root

# (import name, pip name). Colab already ships numpy, pandas, pyarrow,
# scikit-learn, joblib, matplotlib and seaborn, so only these are installed —
# which keeps the bootstrap fast and avoids a "restart runtime" prompt.
COLAB_PACKAGES = [("hydra", "hydra-core"), ("ax", "ax-platform"), ("botorch", "botorch")]

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    checkout = Path("/content") / Path(REPO_URL).stem
    if not checkout.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(checkout)],
            check=True,
        )
    root = checkout / SUBDIR
    missing = [pip for mod, pip in COLAB_PACKAGES if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    # JupyterLab starts the kernel in notebooks/; walk up to the project root.
    root = Path.cwd()
    while not (root / "pyproject.toml").exists() and root != root.parent:
        root = root.parent

os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # makes `import src` work without an editable install

# Extra Hydra overrides consumed by load_config() in section 1. Empty unless
# the Drive cell below fills it in, so local runs are unaffected.
CONFIG_OVERRIDES: list[str] = []

print(f"project root: {root}   colab: {IN_COLAB}")

In [ ]:
# --- Colab: data and artifacts (optional) ----------------------------------
# data/ and models/ are DVC-tracked, so they are not in the Git clone and a
# fresh Colab runtime has neither. Mount Drive and point the config at it —
# Drive also survives a runtime reset, which /content does not.
#
# The scene is the large one: data/external/simulation_map/ holds 3,753 meshes,
# so keep it on Drive rather than re-downloading it per session.
#
# from google.colab import drive
#
# drive.mount("/content/drive")
# DATA_ROOT = "/content/drive/MyDrive/band-tilt/data"
# CONFIG_OVERRIDES += [
#     f"data.mdt_path={DATA_ROOT}/raw/measurement_data.csv",
#     f"data.cell_config_path={DATA_ROOT}/raw/gcell_conf.csv",
#     f"data.scene_file={DATA_ROOT}/external/simulation_map/scene.xml",
#     f"data.train_path={DATA_ROOT}/processed/mdt_train.parquet",
#     f"data.test_path={DATA_ROOT}/processed/mdt_test.parquet",
# ]

## 1. Setup

Compose the config, seed everything, and import from `src/`. Every notebook
starts the same way so that a cell copied between notebooks behaves identically.

In [ ]:
# Standard setup for every notebook in this project.
# Autoreload so edits in src/ take effect without restarting the kernel.
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.config import load_config
from src.utils.plotting import setup_plotting
from src.utils.seed import set_seed

cfg = load_config(overrides=["optim=bo", *CONFIG_OVERRIDES])
set_seed(cfg.seed)
setup_plotting()  # matplotlib/seaborn styling for report-ready figures

pd.set_option("display.max_columns", 50)
cfg

## 2. Search space, surrogate, objective

The objective is the single interface both optimizers score through. Its
`source` decides whether each evaluation is a surrogate prediction or a full
ray-tracing solve.

In [ ]:
from hydra.utils import instantiate

from src.data.load import load_cell_config
from src.optim.objective import Objective
from src.optim.space import TiltSpace
from src.radio import cell_band
from src.surrogate.model import SurrogateMixin

table = cell_band.build_table(load_cell_config(cfg), cfg)
space = TiltSpace(table, cfg)

surrogate = instantiate(cfg.surrogate).load(cfg.surrogate.artifact_path)
objective = Objective(space, cfg, surrogate=surrogate)

print(f"{space.n_dims} dimensions, evaluating against '{cfg.optim.objective.source}'")

## 3. Baseline

Evaluated the same way every candidate will be, so the comparison is like for
like. This is the number the result has to beat.

In [ ]:
theta_0 = space.baseline()
baseline_kpis = objective.kpis(theta_0, source="sionna")
pd.Series(baseline_kpis).to_frame("baseline").loc[list(cfg.kpi.order)]

## 4. Run the loop

`initialize` draws the quasi-random seed design — including the baseline — and
then the model takes over.

Watch the dimensionality. Standard GP-based BO becomes unreliable in the tens of
dimensions, and PROJECT.md section 25.4 asks for exactly this scaling behaviour
to be recorded rather than avoided.

In [ ]:
optimizer = instantiate(cfg.optim, space=space, objective=objective, cfg=cfg)
result = optimizer.run()

print(f"best objective: {result['objective']:.4f}")
print(f"Sionna-RT evaluations consumed: {objective.n_sionna_evaluations}")

## 5. Convergence

The trace, not just the winner. A run that found its best configuration in the
initial random design and never improved has demonstrated that BO added nothing
here — which is a legitimate and useful result, but only if it is visible.

In [ ]:
history = pd.DataFrame(result["history"])
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(history.index, history["objective"], ".", alpha=0.5, label="evaluated")
ax.plot(history.index, history["objective"].cummax(), lw=2, label="best so far")
ax.axvline(cfg.optim.search.n_init, ls="--", lw=1, label="model takes over")
ax.set_xlabel("evaluation")
ax.set_ylabel("objective (larger is better)")
ax.legend()

## 6. Per-KPI trajectory

The scalar objective can improve while hole rate — the highest-priority KPI —
gets worse. That is the specific failure mode of a scalarized encoding of a
lexicographic priority, and this plot is the only place it is visible.

In [ ]:
fig, axes = plt.subplots(1, len(cfg.kpi.order), figsize=(16, 3))
for ax, col in zip(axes, cfg.kpi.order, strict=True):
    ax.plot(history[col], lw=1)
    ax.axhline(baseline_kpis[col], ls="--", lw=1, color="grey")
    ax.set_title(col, fontsize=9)
plt.tight_layout()

## 7. Validate the best candidates with Sionna-RT

**Where a result becomes a claim.** Everything above ran against the surrogate,
and an optimizer given an approximate objective will find the places where that
approximation is most optimistic — that is what optimization does.

Re-run the top `validate_top_k` candidates through Sionna-RT and pick the winner
under the lexicographic priority. Report the gap between prediction and ground
truth: it is itself a result.

In [ ]:
from src.evaluation import validate

validated = validate.validate(result["theta_star"], cfg, predicted=result["predicted_kpis"])
comparison = pd.DataFrame(
    {
        "baseline": baseline_kpis,
        "surrogate prediction": validated["prediction"],
        "Sionna-RT ground truth": validated["ground_truth"],
        "gap": validated["gap"],
    }
).loc[list(cfg.kpi.order)]
comparison

## 8. Result

The tilt table is the deliverable — one row per cell-band, with the offset
derived purely for reporting (PROJECT.md section 27.1, docs/adr/0001).

In [ ]:
from src.evaluation import report

tilts = report.tilt_table(result["theta_star"], table)
record = report.summary(validated, tilts, cfg)
report.export({"bo": record}, "reports/results")
tilts.head(15)

## 9. Handoff checklist

- [ ] The reported KPIs come from Sionna-RT, not the surrogate.
- [ ] The prediction-vs-ground-truth gap is smaller than the claimed improvement.
- [ ] The baseline was evaluated the same way as the candidates.
- [ ] The Sionna-RT evaluation count is recorded for the cost comparison.
- [ ] The run covered every seed in `cfg.optim.seeds`.
- [ ] `theta*`, its KPIs and the full history are saved to `reports/results/`.